In [1]:
import pandas as pd

In [3]:
tabela_usinas = pd.read_parquet("tabela_usinas.parquet")

### Calcula índices de flexibilidade para cada subsistema
- O cálculo é feito para cada subsistema do SIN, considerando 1, 2, 4 e 6 horas de rampa.
- O cálculo considera as fontes solar e eólica de forma negativa (demanda de flexibilidade)
- Enquanto considera outras fontes são consideradas de forma positiva (oferta de flexibilidade).

In [ ]:
filtro_NE = tabela_usinas['id_subsistema'] == 'S'
filtro_REN = tabela_usinas['nom_tipousina'].isin(['EOLIELÉTRICA', 'FOTOVOLTAICA'])
filtro_oferta = filtro_NE & ~filtro_REN
filtro_demanda = filtro_NE & filtro_REN
for horas in [1, 2, 4, 6]:
    cap_instalada = tabela_usinas.loc[filtro_NE, 'P max'].sum()
    flex_oferta = (tabela_usinas.loc[filtro_oferta, "Ind Flex Up P" + str(horas)] * tabela_usinas.loc[filtro_oferta, 'P max']).sum()
    flex_demanda = (-1*tabela_usinas.loc[filtro_demanda, "Ind Flex Down P" + str(horas)] * tabela_usinas.loc[filtro_demanda, 'P max']).sum()
    flex_NE = (flex_oferta + flex_demanda)/cap_instalada
    print(f"NFI do Nordeste é de: {flex_NE:.2f} ")

NFI do Nordeste é de: 0.37 
NFI do Nordeste é de: 0.41 
NFI do Nordeste é de: 0.40 
NFI do Nordeste é de: 0.39 


### Análise com anos de entrada das usinas

In [2]:
capacidade = pd.read_parquet("capacidade_geracao.parquet")

In [9]:
for usina in tabela_usinas["nom_usina"]:
    mask_capacidade = capacidade["nom_usina"].str.upper() == usina.upper()
    if mask_capacidade.any():
        data_entrada = capacidade.loc[mask_capacidade, "dat_entradaoperacao"].values[0]
        tabela_usinas.loc[tabela_usinas["nom_usina"] == usina, "dat_entradaoperacao"] = data_entrada

In [10]:
cols = list(tabela_usinas.columns)
cols.remove('dat_entradaoperacao')
idx = cols.index('P max')
cols = cols[:idx] + ['dat_entradaoperacao'] + cols[idx:]
tabela_usinas = tabela_usinas[cols]

In [14]:
tabela_usinas.loc[tabela_usinas["nom_usina"] == "Alto Jatapu", "P max"] = 10